In [25]:
import json
import base64
import pandas as pd
import time
from pathlib import Path
from PIL import Image
from openai import OpenAI

In [ ]:
# 1. Define your path as a Path object
# Wrapping the string in Path() makes the '/' operator work correctly
dataset_path = Path("/Users/smeh/Desktop/Thesis/DTurtleBench") 

def load_turtle_bench():
    # Now this correctly joins the path parts
    jsonl_file = dataset_path / "dataset.jsonl"
    tasks_dir = dataset_path / "Tasks"
    
    if not jsonl_file.exists():
        print(f"ERROR: Could not find dataset.jsonl at {jsonl_file}")
        print(f"Current search path: {jsonl_file}")
        return None

    data = []
    with open(jsonl_file, 'r', encoding='utf-8') as f:
        for line in f:
            entry = json.loads(line)
            
            # TurtleBench structure: Tasks/[task_id]/image/[task_id].png
            task_id = str(entry['id'])
            
            # Joining subpaths using the Path object '/' operator
            entry['image_path'] = str(tasks_dir / task_id / "image" / f"{task_id}.png")
            entry['ground_truth_path'] = str(tasks_dir / task_id / "QA" / "ground_truth.py")
            
            data.append(entry)
    
    return pd.DataFrame(data)

# 2. Execute and Explore
df = load_turtle_bench()

if df is not None:
    print(f"Successfully loaded {len(df)} TurtleBench tasks.")
    
    # Preview a specific task
    sample_task = df.iloc[0]
    print(f"\nTask ID: {sample_task['id']}")
    print(f"Query: {sample_task['query']}")
    
    # Display the image using PIL
    try:
        img = Image.open(sample_task['image_path'])
        img.show()
    except Exception as e:
        print(f"Could not open image: {e}")

Successfully loaded 260 TurtleBench tasks.

Task ID: 1
Query: recreates the given shape + its horizontal and vertical diameters.


In [ ]:
# 1. Configuration
dataset_path = Path("/Users/smeh/Desktop/Thesis/DTurtleBench")
NGROK_URL = "https://606c61eefd92.ngrok-free.app/v1"

# Specify the model you want to test (Ensure this matches 'ollama list' on remote PC)
MODEL_NAME = "llava:7b" 

client = OpenAI(
    base_url=NGROK_URL,
    api_key="ollama",
    timeout=180.0 # High timeout for remote visual reasoning
)

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def get_turtle_code_response(model_name, original_query, image_path):
    base64_image = encode_image(image_path)
    
    system_msg = "You are a specialized Python developer. Your goal is to write clean Python Turtle code that draws the pattern in the image."
    user_prompt = f"Original Question: {original_query}\n\nBased on the image, write the Python Turtle code to reproduce this. Provide only the code."

    # Record the start time for the API call
    start_time = time.time()

    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_msg},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": user_prompt},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{base64_image}"
                        }
                    },
                ],
            }
        ],
        max_tokens=1000,
        temperature=0.1
    )
    
    # Calculate duration
    end_time = time.time()
    duration = end_time - start_time
    
    return response.choices[0].message.content, duration

def load_turtle_bench():
    jsonl_file = dataset_path / "dataset.jsonl"
    tasks_dir = dataset_path / "Tasks"
    if not jsonl_file.exists(): return None

    data = []
    with open(jsonl_file, 'r', encoding='utf-8') as f:
        for line in f:
            entry = json.loads(line)
            task_id = str(entry['id'])
            entry['image_path'] = str(tasks_dir / task_id / "image" / f"{task_id}.png")
            data.append(entry)
    return pd.DataFrame(data)

# 2. Execution Logic
df = load_turtle_bench()

if df is not None:
    # Testing the first task (Index 0)
    sample = df.iloc[0]
    
    print("\n" + "="*60)
    print(f"🚀 RUNNING TASK ID: {sample['id']}")
    print(f"📝 QUERY: {sample['query']}")
    print("="*60)

    # Automatically open the visual reference for the user
    try:
        print("Opening input image for review...")
        img = Image.open(sample['image_path'])
        img.show()
    except Exception as e:
        print(f"Warning: Could not open image: {e}")

    print(f"📡 Sending request to Remote PC...")
    
    try:
        # Call model and get both the code and the time it took
        model_result, run_time = get_turtle_code_response(MODEL_NAME, sample['query'], sample['image_path'])
        
        print("\n" + "-"*40)
        print("📊 TASK BENCHMARK RESULTS")
        print(f"🤖 Model Used: {MODEL_NAME}")
        print(f"⏱️  Total Run Time: {run_time:.2f} seconds")
        print("-" * 40)
        
        print("\n✅ GENERATED TURTLE CODE:")
        print(model_result)
        
    except Exception as e:
        print(f"\n❌ API CONNECTION ERROR: {e}")


🚀 RUNNING TASK ID: 1
📝 QUERY: creates exactly the same shape.
Opening input image for review...
📡 Sending request to Remote PC...

----------------------------------------
📊 TASK BENCHMARK RESULTS
🤖 Model Used: llava:7b
⏱️  Total Run Time: 9.38 seconds
----------------------------------------

✅ GENERATED TURTLE CODE:
 The image you've provided appears to be a simple geometric figure consisting of a circle with a line segment connecting two points on its circumference. To recreate this shape using Python's turtle module, we can use the following code:

```python
import turtle

# Set up the turtle screen and pen
t = turtle.Turtle()
t.speed(0)  # Set the speed to slowest for better precision

# Draw the circle
t.penup()  # Lift the pen
t.circle(50)  # Draw a circle with radius 50
t.pendown()  # Lower the pen

# Draw the line segment
t.penup()
t.goto(-40, -20)  # Move to the starting point of the line segment
t.pendown()
t.forward(80)  # Draw a forward movement of 80 units
t.right(90)  #

In [28]:
# 1. Configuration
dataset_path = Path("/Users/smeh/Desktop/Thesis/DTurtleBench")
NGROK_URL = "https://606c61eefd92.ngrok-free.app/v1"

# Specify the model you want to test
MODEL_NAME = "llava:7b" 

client = OpenAI(
    base_url=NGROK_URL,
    api_key="ollama",
    timeout=180.0 
)

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def get_turtle_code_response(model_name, original_query, image_path):
    base64_image = encode_image(image_path)
    
    # SYSTEM MSG: Strict instruction for code only
    system_msg = "You are a specialized Python developer. Your goal is to write clean Python Turtle code."
    
    # UPDATED PROMPT: Explicitly skips setup
    user_prompt = (
        f"Original Question: {original_query}\n\n"
        "INSTRUCTION: Write the Python Turtle code to draw this pattern based on the image.\n"
        "IMPORTANT RULES:\n"
        "1. Assume the turtle object 't' is already created.\n"
        "2. DO NOT write setup code like 'import turtle', 't = turtle.Turtle()', or 'screen = ...'.\n"
        "3. Provide ONLY the drawing logic loops and commands.\n"
    )

    start_time = time.time()

    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_msg},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": user_prompt},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{base64_image}"
                        }
                    },
                ],
            }
        ],
        max_tokens=1000,
        temperature=0.1
    )
    
    end_time = time.time()
    duration = end_time - start_time
    
    return response.choices[0].message.content, duration

def load_turtle_bench():
    jsonl_file = dataset_path / "dataset.jsonl"
    tasks_dir = dataset_path / "Tasks"
    if not jsonl_file.exists(): return None

    data = []
    with open(jsonl_file, 'r', encoding='utf-8') as f:
        for line in f:
            entry = json.loads(line)
            task_id = str(entry['id'])
            entry['image_path'] = str(tasks_dir / task_id / "image" / f"{task_id}.png")
            data.append(entry)
    return pd.DataFrame(data)

# 2. Execution Logic
df = load_turtle_bench()

if df is not None:
    # Loop through the first 3 tasks
    tasks_to_run = df.head(3)
    
    for index, sample in tasks_to_run.iterrows():
        print("\n" + "="*60)
        print(f"🚀 RUNNING TASK {index + 1}/3 | ID: {sample['id']}")
        print(f"📝 QUERY: {sample['query']}")
        print("="*60)

        # Open image for review
        try:
            print(f"Opening image for Task {sample['id']}...")
            img = Image.open(sample['image_path'])
            img.show()
        except Exception as e:
            print(f"Warning: Could not open image: {e}")

        print(f"📡 Sending request to Remote PC...")
        
        try:
            model_result, run_time = get_turtle_code_response(MODEL_NAME, sample['query'], sample['image_path'])
            
            print("\n" + "-"*40)
            print("📊 TASK BENCHMARK RESULTS")
            print(f"🤖 Model: {MODEL_NAME}")
            print(f"⏱️  Time:  {run_time:.2f} seconds")
            print("-" * 40)
            
            print("\n✅ GENERATED LOGIC (NO SETUP):")
            print(model_result)
            
        except Exception as e:
            print(f"\n❌ API CONNECTION ERROR: {e}")

        # Wait 5 seconds before the next task (if not the last one)
        if index < len(tasks_to_run) - 1:
            print("\n⏳ Waiting 5 seconds before next task...")
            time.sleep(5)


🚀 RUNNING TASK 1/3 | ID: 1
📝 QUERY: recreates the given shape + its horizontal and vertical diameters.
Opening image for Task 1...
📡 Sending request to Remote PC...

----------------------------------------
📊 TASK BENCHMARK RESULTS
🤖 Model: llava:7b
⏱️  Time:  8.84 seconds
----------------------------------------

✅ GENERATED LOGIC (NO SETUP):
 The image you provided appears to be a simple geometric shape, possibly a circle with a smaller circle inside it. To recreate this pattern using Python Turtle, we can use the following code:

```python
import turtle

# Create a new turtle object
t = turtle.Turtle()

# Set up the drawing area
t.speed(0)  # Set the speed to slowest for precise movements
t.bgcolor("white")  # Set the background color to white

# Draw the outer circle
t.penup()  # Lift the pen
t.circle(50)  # Draw a circle with radius 50
t.pendown()  # Lower the pen

# Draw the inner circle
t.penup()
t.circle(30)
t.pendown()

# Move to the center of the circles
t.penup()
t.goto(0, 